# 台股 ML 預測 — FT-Transformer 訓練

**模型**: FT-Transformer（PyTorch 手刻）
**任務**: 二元分類 × 2（做多模型 / 放空模型）
**不平衡處理**: SMOTE-NC + Tomek Links（訓練集）
**評估**: Top-K Precision（per rev_date 橫斷面）
**時間切分**:
```
Train : 2012, 2015, 2018, 2021, 2022, 2023
Val   : 2013, 2016, 2019
Test  : 2014, 2017, 2020, 2024
```

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install imbalanced-learn -q

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 載入資料

In [ ]:
X        = pd.read_parquet('X_features.parquet')
y_top    = pd.read_parquet('y_top.parquet').squeeze()
y_bottom = pd.read_parquet('y_bottom.parquet').squeeze()
y_return = pd.read_parquet('y_return.parquet').squeeze()

# 欄位分類
bins_cols = [c for c in X.columns if c.endswith('_bins')]
cont_cols = [c for c in X.columns if not c.endswith('_bins')]

cont_indices = [X.columns.get_loc(c) for c in cont_cols]
cat_indices  = [X.columns.get_loc(c) for c in bins_cols]

n_cont           = len(cont_cols)
cat_cardinalities = [6] * len(bins_cols)  # 所有 bins 皆為 0~5，共 6 類

print(f'連續特徵: {n_cont}　類別特徵: {len(bins_cols)}')
print(f'總樣本數: {len(X):,}　日期數: {X.index.get_level_values("datetime").nunique()}')

# 確認實際資料涵蓋的年份
available_years = sorted(X.index.get_level_values('datetime').year.unique())
print(f'資料年份: {available_years}')

## 時間切分

In [ ]:
TRAIN_YEARS = [2012, 2015, 2018, 2021, 2022, 2023]
VAL_YEARS   = [2013, 2016, 2019]
TEST_YEARS  = [2014, 2017, 2020, 2024]

years = X.index.get_level_values('datetime').year

train_mask = years.isin(TRAIN_YEARS)
val_mask   = years.isin(VAL_YEARS)
test_mask  = years.isin(TEST_YEARS)

X_train, X_val, X_test         = X[train_mask], X[val_mask], X[test_mask]
y_top_train, y_top_val, y_top_test       = y_top[train_mask], y_top[val_mask], y_top[test_mask]
y_bot_train, y_bot_val, y_bot_test       = y_bottom[train_mask], y_bottom[val_mask], y_bottom[test_mask]
y_ret_train, y_ret_val, y_ret_test       = y_return[train_mask], y_return[val_mask], y_return[test_mask]

for name, y in [('Train', y_top_train), ('Val', y_top_val), ('Test', y_top_test)]:
    print(f'{name:5s}: {len(y):>7,} 樣本  正類={y.sum():>5,.0f} ({y.mean():.2%})')

## SMOTE-NC + Tomek Links

- `SMOTENC`：對連續特徵插值生成合成少數樣本，類別特徵（bins）用 k-NN 鄰居的眾數填入
- `TomekLinks`：移除靠近決策邊界的多數類樣本，清潔邊界
- 只套用在**訓練集**，val / test 維持原始分布
- 兩個模型各自獨立 resample（label 不同，resample 結果不同）

In [ ]:
def apply_smote_tomek(X_df: pd.DataFrame, y: pd.Series,
                      cat_indices: list, random_state: int = 42):
    """
    對訓練集套用 SMOTE-NC + TomekLinks。
    回傳 numpy arrays（無 MultiIndex）。
    """
    X_np = X_df.values.astype(float)
    y_np = y.values.astype(int)

    print(f'  Resampling 前: 正類={y_np.sum():,}  負類={(1-y_np).sum():,}')

    smote = SMOTENC(
        categorical_features=cat_indices,
        k_neighbors=5,
        random_state=random_state,
    )
    smt = SMOTETomek(smote=smote, random_state=random_state)

    X_res, y_res = smt.fit_resample(X_np, y_np)

    print(f'  Resampling 後: 正類={y_res.sum():,}  負類={(1-y_res).sum():,}')
    return X_res, y_res.astype(float)

In [ ]:
print('=== Top model (做多) 訓練集 resample ===')
X_top_res, y_top_res = apply_smote_tomek(X_train, y_top_train, cat_indices)

print('\n=== Bottom model (放空) 訓練集 resample ===')
X_bot_res, y_bot_res = apply_smote_tomek(X_train, y_bot_train, cat_indices)

## Dataset & DataLoader

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X_np: np.ndarray, y_np: np.ndarray,
                 cont_indices: list, cat_indices: list):
        self.X_cont = torch.FloatTensor(X_np[:, cont_indices])
        self.X_cat  = torch.LongTensor(X_np[:, cat_indices].astype(int))
        self.y      = torch.FloatTensor(y_np)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_cont[idx], self.X_cat[idx], self.y[idx]


def make_loader(X_np, y_np, cont_indices, cat_indices, batch_size, shuffle=True):
    ds = TabularDataset(X_np, y_np, cont_indices, cat_indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=True)

## FT-Transformer（PyTorch 手刻）

**架構**
```
x_cont, x_cat
  ↓ FeatureTokenizer
    連續: x_j × W_j + b_j  →  (B, n_cont, d)
    類別: Embedding[i](x_cat[:,i])  →  (B, n_cat, d)
    concat  →  (B, n_feat, d)
  ↓ prepend [CLS] token  →  (B, 1+n_feat, d)
  ↓ TransformerEncoder（Pre-LN, GELU）× n_layers
  ↓ 取 CLS output[:, 0, :]  →  (B, d)
  ↓ LayerNorm → Linear → GELU → Dropout → Linear
  →  logit (B,)
```

In [ ]:
class FeatureTokenizer(nn.Module):
    def __init__(self, n_cont: int, cat_cardinalities: list, d_token: int):
        super().__init__()
        # 每個連續特徵有自己的 weight 向量和 bias 向量
        self.cont_W = nn.Parameter(torch.empty(n_cont, d_token))
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d_token))
        nn.init.kaiming_uniform_(self.cont_W, a=math.sqrt(5))

        # 每個類別特徵有自己的 Embedding table
        self.cat_emb = nn.ModuleList([
            nn.Embedding(card, d_token)
            for card in cat_cardinalities
        ])

    def forward(self, x_cont: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        # x_cont: (B, n_cont)  float
        # x_cat:  (B, n_cat)   long
        t_cont = x_cont.unsqueeze(-1) * self.cont_W + self.cont_b   # (B, n_cont, d)
        t_cat  = torch.stack(
            [self.cat_emb[i](x_cat[:, i]) for i in range(x_cat.shape[1])],
            dim=1
        )                                                             # (B, n_cat, d)
        return torch.cat([t_cont, t_cat], dim=1)                     # (B, n_feat, d)


class FTTransformer(nn.Module):
    def __init__(
        self,
        n_cont: int,
        cat_cardinalities: list,
        d_token: int = 192,
        n_heads: int = 8,
        n_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_token % n_heads == 0, 'd_token 必須能被 n_heads 整除'

        self.tokenizer = FeatureTokenizer(n_cont, cat_cardinalities, d_token)

        # 可學習的 [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Pre-LN Transformer（比 Post-LN 訓練更穩定）
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # 輸出頭（接在 CLS token 上）
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token // 2, 1),
        )

    def forward(self, x_cont: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        tokens = self.tokenizer(x_cont, x_cat)                        # (B, n_feat, d)
        cls    = self.cls_token.expand(tokens.size(0), -1, -1)        # (B, 1, d)
        tokens = torch.cat([cls, tokens], dim=1)                      # (B, 1+n_feat, d)
        out    = self.transformer(tokens)                              # (B, 1+n_feat, d)
        return self.head(out[:, 0]).squeeze(-1)                        # (B,)  logit

## 評估指標 — Top-K Precision

每個 rev_date 獨立計算：預測分數最高的 K 支股票中，真正屬於前（後）1% 的比例。  
K 預設等於當日股票數的 1%（與 label 定義對齊），也可指定固定 K。

In [ ]:
def top_k_precision(
    X_df: pd.DataFrame,
    y_series: pd.Series,
    model: nn.Module,
    cont_indices: list,
    cat_indices: list,
    k_pct: float = 0.01,
    batch_size: int = 1024,
    device=DEVICE,
) -> float:
    """
    Per rev_date 計算 Top-K Precision，回傳所有日期的平均值。

    k_pct=0.01 → 每天取預測分數最高的 1% 股票，計算其中有幾支是真正的正類。
    """
    model.eval()
    X_np = X_df.values.astype(float)

    # 批次推論（避免 OOM）
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xc = torch.FloatTensor(X_np[i:i+batch_size, cont_indices]).to(device)
            xk = torch.LongTensor(X_np[i:i+batch_size, cat_indices].astype(int)).to(device)
            logits = model(xc, xk)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)

    probs_all = np.concatenate(all_probs)

    # 組合成 DataFrame，按 datetime 分組
    result_df = pd.DataFrame({
        'prob' : probs_all,
        'label': y_series.values,
    }, index=X_df.index)

    precisions = []
    for dt, grp in result_df.groupby(level='datetime'):
        k = max(1, int(np.ceil(len(grp) * k_pct)))
        top_k = grp.nlargest(k, 'prob')
        precisions.append(top_k['label'].mean())

    return float(np.mean(precisions))

## 訓練與驗證函數

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x_cont, x_cat, y in loader:
        x_cont, x_cat, y = x_cont.to(device), x_cat.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x_cont, x_cat)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


def train_model(
    model_name: str,
    X_train_res: np.ndarray,
    y_train_res: np.ndarray,
    X_val_df: pd.DataFrame,
    y_val: pd.Series,
    cont_indices: list,
    cat_indices: list,
    cat_cardinalities: list,
    config: dict,
    device=DEVICE,
) -> nn.Module:

    # pos_weight 從 resampled 資料動態計算
    n_pos = y_train_res.sum()
    n_neg = len(y_train_res) - n_pos
    pos_w = torch.tensor(n_neg / n_pos, dtype=torch.float32).to(device)
    print(f'[{model_name}] pos_weight = {pos_w.item():.3f}  '
          f'(neg={n_neg:,} / pos={n_pos:,})')

    loader = make_loader(
        X_train_res, y_train_res,
        cont_indices, cat_indices,
        batch_size=config['batch_size'],
    )

    model = FTTransformer(
        n_cont=len(cont_indices),
        cat_cardinalities=cat_cardinalities,
        d_token=config['d_token'],
        n_heads=config['n_heads'],
        n_layers=config['n_layers'],
        dropout=config['dropout'],
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['epochs']
    )

    best_val_prec = -1.0
    patience_cnt  = 0
    best_state    = None

    for epoch in range(1, config['epochs'] + 1):
        train_loss = train_one_epoch(model, loader, optimizer, criterion, device)
        scheduler.step()

        val_prec = top_k_precision(
            X_val_df, y_val, model,
            cont_indices, cat_indices,
            k_pct=config['k_pct'],
            device=device,
        )

        improved = val_prec > best_val_prec
        if improved:
            best_val_prec = val_prec
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
        else:
            patience_cnt += 1

        if epoch % 5 == 0 or improved:
            flag = ' ✅' if improved else ''
            print(f'  Epoch {epoch:3d} | loss={train_loss:.4f} | '
                  f'val Top-K Prec={val_prec:.4f}{flag}')

        if patience_cnt >= config['patience']:
            print(f'  Early stopping at epoch {epoch}')
            break

    # 載入最佳權重
    model.load_state_dict(best_state)
    print(f'  Best val Top-K Precision: {best_val_prec:.4f}')
    return model

## 訓練設定

In [ ]:
CONFIG = {
    # 模型架構
    'd_token'      : 192,
    'n_heads'      : 8,
    'n_layers'     : 3,
    'dropout'      : 0.1,
    # 訓練
    'lr'           : 1e-4,
    'weight_decay' : 1e-5,
    'batch_size'   : 256,
    'epochs'       : 100,
    'patience'     : 10,
    # 評估
    'k_pct'        : 0.01,   # Top-K Precision：每日前 1%
}

## 訓練 Top Model（做多訊號）

In [ ]:
print('=' * 55)
print('  TOP MODEL — 預測報酬前 1%（做多）')
print('=' * 55)

model_top = train_model(
    model_name        = 'Top',
    X_train_res       = X_top_res,
    y_train_res       = y_top_res,
    X_val_df          = X_val,
    y_val             = y_top_val,
    cont_indices      = cont_indices,
    cat_indices       = cat_indices,
    cat_cardinalities = cat_cardinalities,
    config            = CONFIG,
    device            = DEVICE,
)

## 訓練 Bottom Model（放空訊號）

In [ ]:
print('=' * 55)
print('  BOTTOM MODEL — 預測報酬後 1%（放空）')
print('=' * 55)

model_bot = train_model(
    model_name        = 'Bottom',
    X_train_res       = X_bot_res,
    y_train_res       = y_bot_res,
    X_val_df          = X_val,
    y_val             = y_bot_val,
    cont_indices      = cont_indices,
    cat_indices       = cat_indices,
    cat_cardinalities = cat_cardinalities,
    config            = CONFIG,
    device            = DEVICE,
)

## Test 最終評估

In [ ]:
test_top_prec = top_k_precision(
    X_test, y_top_test, model_top,
    cont_indices, cat_indices, k_pct=CONFIG['k_pct'],
)
test_bot_prec = top_k_precision(
    X_test, y_bot_test, model_bot,
    cont_indices, cat_indices, k_pct=CONFIG['k_pct'],
)

# 隨機基準：若隨機選 1%，期望 precision = 1%
baseline = CONFIG['k_pct']

print('=' * 45)
print('  TEST SET 最終結果')
print('=' * 45)
print(f'  Top    model | Top-K Prec = {test_top_prec:.4f}  (baseline={baseline:.4f})')
print(f'  Bottom model | Top-K Prec = {test_bot_prec:.4f}  (baseline={baseline:.4f})')
print(f'  Top    lift = {test_top_prec/baseline:.2f}x')
print(f'  Bottom lift = {test_bot_prec/baseline:.2f}x')

## 儲存模型

In [ ]:
def save_model(model, path, cont_cols, bins_cols, config):
    torch.save({
        'state_dict'       : model.state_dict(),
        'cont_cols'        : cont_cols,
        'bins_cols'        : bins_cols,
        'cat_cardinalities': cat_cardinalities,
        'config'           : config,
    }, path)
    print(f'✅ 儲存: {path}')

save_model(model_top, 'model_top.pt',    cont_cols, bins_cols, CONFIG)
save_model(model_bot, 'model_bottom.pt', cont_cols, bins_cols, CONFIG)